# KdV on a nonperiodic finite interval — JAX BSPF

Solve $u_t+6uu_x+u_{xxx}=0$ on $[-6,6]$ using the traveling soliton
$$u(x,t)=\frac{c}{2}\operatorname{sech}^2\!\left[\frac{\sqrt c}{2}(x-x_0-ct)\right],
\qquad c=2,\quad x_0=-2.$$
The soliton supplies initial data and **nonzero, time-dependent boundary data**.
The interval is deliberately short: its endpoint mismatch reaches about 0.014,
so this is not a periodic test or a negligible-tail approximation.

Install `python -m pip install -e '.[host,notebook,test]' -e './packages/models[precision,test]'`. Use an x64 kernel and,
on the affected OpenMP BLAS installation, `OMP_NUM_THREADS=1` before Python
startup or the corrected-BLAS launcher.

In [ ]:
import bspf_models.waves.kdv as bspf_kdv
import pybspf.operators as bspf_operators
import pybspf.plans as bspf_plans

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pybspf as bspf


## Boundary conditions and model setup

For this sign of dispersion prescribe $u(-6,t)$, $u(6,t)$, and $u_x(6,t)$.
`plan_kdv` strongly lifts the two endpoint values and supplies the right slope
as a natural weak boundary load. Its actual slope residual is checked below.
The lifting includes the time derivatives of the Dirichlet data via JAX JVP.

Resolved Gauss quadrature integrates the BSPF trial functions and nonlinear
flux. The homogeneous linear weak operator is dissipative in the mass norm.
`integrate_kdv` uses matrix-function ETDRK4 rather than diagonalizing the
strongly nonnormal third-derivative operator. Output times must be uniform.
Stiff boundary forcing can reduce observed temporal order, so refinement is
measured rather than inferred from the formal fourth order.

In [ ]:
speed, x0 = 2., -2.
ends = jnp.array([-6., 6.])
times = jnp.linspace(0., 2., 21)

def reference(x, t):
    z = jnp.sqrt(speed)/2*(x-x0-speed*t)
    return speed/2/jnp.cosh(z)**2

def boundary(t):
    values = reference(ends, t)
    z_right = jnp.sqrt(speed)/2*(ends[1]-x0-speed*t)
    slope_right = -jnp.sqrt(speed)*values[1]*jnp.tanh(z_right)
    return jnp.array([values[0], values[1], slope_right])

def setup(n, quadrature_order=8):
    x = jnp.linspace(ends[0], ends[1], n)
    spatial = bspf_plans.plan_1d(x, degree=7, n_basis=32, boundary_points=9)
    return x, spatial, bspf_kdv.plan_kdv(spatial, quadrature_order=quadrature_order)

def evolve(plan, x, substeps):
    return jax.jit(lambda: bspf_kdv.integrate_kdv(
        plan, reference(x, 0.), times, boundary=boundary, substeps=substeps))()

x, spatial, plan = setup(129)
solution = evolve(plan, x, 640)  # dt = 0.00015625
larger_step = evolve(plan, x, 320)
coarse_x, _, coarse_plan = setup(65)
coarse = evolve(coarse_plan, coarse_x, 640)
_, _, quadrature_plan = setup(129, quadrature_order=10)
quadrature_check = evolve(quadrature_plan, x, 640)


## Field, boundary, and integral checks

Mass and $\int u^2 dx$ need **not** be constant with these boundary data:
$\frac{d}{dt}\int u\,dx=-[3u^2+u_{xx}]_{-6}^{6}$.
Compare their whole histories with analytic finite-interval integrals instead
of incorrectly asserting conservation. Values at both endpoints are imposed
exactly; the weak right-slope condition has a separately measured residual.
No periodic extension, padding, filtering, or solution repair is used.

In [ ]:
exact = reference(x[None, :], times[:, None])
prescribed = jax.vmap(boundary)(times)
field_error = jnp.max(jnp.abs(solution-exact), axis=1)
coarse_error = jnp.max(jnp.abs(coarse-reference(coarse_x[None, :], times[:, None])))
step_error = jnp.max(jnp.abs(larger_step-exact))
quadrature_error = jnp.max(jnp.abs(solution-quadrature_check))
value_error = jnp.max(jnp.abs(solution[:, jnp.array([0, -1])]-prescribed[:, :2]))
slope = bspf_operators.differentiate(spatial, solution.T)[-1]
slope_error = jnp.max(jnp.abs(slope-prescribed[:, 2]))
endpoint_mismatch = jnp.max(jnp.abs(prescribed[:, 0]-prescribed[:, 1]))
qfield = (solution[:, plan.free]@plan.values.T
          +prescribed[:, :2]@plan.boundary_values.T)
mass = qfield@plan.quadrature_weights
quadratic = qfield**2@plan.quadrature_weights
z = jnp.sqrt(speed)/2*(ends[None, :]-x0-speed*times[:, None])
s = jnp.tanh(z)
exact_mass = jnp.sqrt(speed)*(s[:, 1]-s[:, 0])
primitive = s-s**3/3
exact_quadratic = speed**1.5/2*(primitive[:, 1]-primitive[:, 0])
mass_error = jnp.max(jnp.abs(mass-exact_mass))
quadratic_error = jnp.max(jnp.abs(quadratic-exact_quadratic))
print(f"field error: {field_error.max():.3e}; coarse grid: {coarse_error:.3e}; doubled step: {step_error:.3e}")
print(f"quadrature difference: {quadrature_error:.3e}")
print(f"endpoint mismatch: {endpoint_mismatch:.3e}; value error: {value_error:.3e}; right-slope error: {slope_error:.3e}")
print(f"mass error: {mass_error:.3e}; quadratic-integral error: {quadratic_error:.3e}")
assert jnp.all(jnp.isfinite(solution))
assert field_error.max() < 2e-9
assert field_error.max() < coarse_error/20
assert field_error.max() < step_error/2
assert quadrature_error < 1e-9
assert endpoint_mismatch > 1e-2
assert jnp.max(jnp.abs(prescribed[:, 2])) > 1e-2
assert value_error < 1e-14
assert slope_error < 2e-8
assert mass_error < 5e-9
assert quadratic_error < 5e-9


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for i in (0, 10, 20):
    axes[0, 0].plot(x, solution[i], label=f"JAX t={float(times[i]):g}")
    axes[0, 0].plot(x, exact[i], "k--", linewidth=0.8)
axes[0, 0].set(xlabel="x", ylabel="u", title="Nonperiodic KdV soliton (dashed: exact)")
axes[0, 0].legend()
axes[0, 1].plot(times, prescribed[:, 0], label="u(left)")
axes[0, 1].plot(times, prescribed[:, 1], label="u(right)")
axes[0, 1].plot(times, prescribed[:, 2], label="u_x(right)")
axes[0, 1].set(xlabel="t", title="Nonzero, time-dependent boundary data")
axes[0, 1].legend()
axes[1, 0].semilogy(times[1:], field_error[1:], label="Fine time step")
axes[1, 0].semilogy(times[1:], jnp.max(jnp.abs(larger_step-exact), axis=1)[1:], label="Doubled step")
axes[1, 0].set(xlabel="t", ylabel="Max field error", title="Time refinement")
axes[1, 0].legend()
axes[1, 1].plot(times, mass, label="JAX mass")
axes[1, 1].plot(times, exact_mass, "k--", label="Exact finite-interval mass")
axes[1, 1].set(xlabel="t", ylabel="Integral of u", title="Boundary flux changes the mass")
axes[1, 1].legend()
plt.show()
